# 03. Ranking Analysis: What Is Highest and Lowest?
### Primary Analytical Purpose: *What is highest, what is lowest, and in what order?*
**Lecture Reference:** Purpose 3 — Ranking (Slides 17–20, 45)

Ranking analysis orders entities from greatest to least (or least to greatest) to determine market leaders, evaluate laggards, and observe the shape of the distribution curve. It addresses questions such as:
- Which product, destination, or branch leads the ranking?
- How wide is the gap between the #1 and #2 positions?
- Does the ranking drop off sharply or taper into a long tail?

---

## 1. Scenario and Dataset Preview
**Business Scenario:** Qatar Tourism & Commercial Real Estate tracks the annual footfall (in millions of visitors) across 8 major commercial and cultural destinations in Qatar: *Place Vendôme*, *Doha Festival City*, *Mall of Qatar*, *Villaggio Mall*, *Souq Waqif*, *City Center Doha*, *Katara Cultural Village*, and *The Pearl Island*.

The dataset is loaded from `data/qatar_destinations_footfall.csv`. Let us inspect the data.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Set modern Seaborn theme
sns.set_theme(style="whitegrid")

# Load footfall dataset
df = pd.read_csv("data/qatar_destinations_footfall.csv")

# Display the dataset
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df

## 2. Primary Visualization: Sorted Horizontal Bar Chart (The Default)
- **What the chart shows:** Entities ordered descending from highest to lowest value, rendered horizontally.
- **When it is useful:** Answering 'what is highest, what is lowest, and in what order?'.
- **Why horizontal (Slide 17 & 19):** Horizontal bars allow destination names to read naturally from left to right without diagonal tilting or neck craning.
- **Key principle (Slide 19 & 46):** **Sort descending; never leave default alphabetical order.** Sorting does the analytical work!

In [ ]:
# Principle 6: Sort by value descending
df_sorted = df.sort_values("annual_visitors_millions", ascending=False)

# --- Seaborn Implementation: Sorted Horizontal Bar Chart ---
plt.figure(figsize=(8.5, 4.8))

ax = sns.barplot(
    data=df_sorted, 
    y="destination", 
    x="annual_visitors_millions", 
    color="#C77E23"  # Single intentional warm amber color
)

# Principle 4: Honest zero baseline
ax.set_xlim(0, 22)

# Principle 2: Title states the analytical finding
ax.set_title("Place Vendôme Leads Footfall at 18.5M Annual Visitors", fontsize=13, weight="bold")
ax.set_xlabel("Annual Visitors (Millions)", fontsize=11)
ax.set_ylabel("", fontsize=11)

# Direct data labels
for p in ax.patches:
    val = p.get_width()
    ax.annotate(f"{val:.1f}M", (val + 0.3, p.get_y() + p.get_height() / 2.), 
                va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly Express Implementation: Interactive Horizontal Bar Chart ---
# In Plotly horizontal bars, categoryorder='total ascending' puts the highest bar at the top!
fig = px.bar(
    df, 
    x="annual_visitors_millions", 
    y="destination", 
    orientation="h",
    text_auto=".1f",
    title="Place Vendôme Leads Footfall at 18.5M Annual Visitors",
    labels={"annual_visitors_millions": "Annual Visitors (Millions)", "destination": ""}
)

fig.update_yaxes(categoryorder="total ascending")
fig.update_traces(marker_color="#C77E23")
fig.update_xaxes(range=[0, 22])
fig.show()

### Structured Interpretation: The 5-Step Framework
1. **Question:** Which commercial and leisure destinations attract the highest footfall in Qatar, and where does the distribution flatten?
2. **Visualization:** Sorted horizontal bar chart with shared zero baseline.
3. **Observation:** Place Vendôme leads with 18.5M visitors, followed by Doha Festival City (16.2M) and Mall of Qatar (14.8M). The top 3 capture 49.5M total visits. Souq Waqif leads cultural sites at 9.2M, while The Pearl Island recorded 6.1M.
4. **Insight:** Commercial retail-entertainment mega-malls capture the vast majority of volume, while outdoor cultural/leisure hubs occupy a secondary tier.
5. **Implication:** Major nationwide brand campaigns and festival pop-ups should secure flagship space in the top 3 malls to maximize footfall reach.

## 3. Alternative 1: Top-N Chart with an Explicit 'Other' Bar
- **What the chart shows:** The top 3 leaders highlighted individually, with all remaining items aggregated into a single 'Other' bar.
- **When to use instead:** When executive stakeholders need to focus strictly on leaders, but omitting the rest would deceptively hide total volume.
- **Rule of thumb (Slide 17 & 19):** **Always add an 'Other' bar so the reader knows what was excluded.** Showing five of forty without 'Other' is a deceptive hidden cut.

In [ ]:
# Extract Top 3 and aggregate remainder into 'Other'
top_n = 3
df_top = df_sorted.head(top_n).copy()
other_val = df_sorted.iloc[top_n:]["annual_visitors_millions"].sum()

other_row = pd.DataFrame([{
    "destination": f"Other ({len(df_sorted) - top_n} Destinations)",
    "category": "Aggregated",
    "annual_visitors_millions": round(other_val, 1)
}])

df_top_other = pd.concat([df_top, other_row], ignore_index=True)

# --- Seaborn: Top-N with Other Bar ---
plt.figure(figsize=(8, 3.8))

# Use distinct color for leaders and muted grey for 'Other'
bar_colors = ["#C77E23"] * top_n + ["#888888"]
ax_top = sns.barplot(
    data=df_top_other, 
    y="destination", 
    x="annual_visitors_millions", 
    palette=bar_colors
)

ax_top.set_xlim(0, 50)
ax_top.set_title("Top 3 Destinations vs. All Other 5 Destinations Combined", fontsize=12, weight="bold")
ax_top.set_xlabel("Annual Visitors (Millions)", fontsize=11)
ax_top.set_ylabel("", fontsize=11)

for p in ax_top.patches:
    val = p.get_width()
    ax_top.annotate(f"{val:.1f}M", (val + 0.6, p.get_y() + p.get_height() / 2.), 
                    va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### Interpretation for Top-N Chart
- **Observation:** The top 3 destinations account for 49.5M visitors, while the remaining 5 combined account for 44.8M visitors.
- **Derived Insight:** Showing the 'Other' bar makes it immediately evident that while the top 3 lead individually, the long tail represents nearly half of Qatar's total commercial footfall.

## 4. Alternative 2: Lollipop Chart (Minimal Ink for Many Items)
- **What the chart shows:** A thin stem line connecting the zero baseline to a circle marker at each value.
- **When to use instead:** When ranking 15–30 items (Slide 17 & 45).
- **Advantage:** Same analytical information as a bar chart, with far less graphical clutter and ink.

In [ ]:
# --- Matplotlib / Seaborn: Lollipop Chart ---
plt.figure(figsize=(8.5, 4.5))

# Plot stem lines from 0 to value
plt.hlines(
    y=df_sorted["destination"], 
    xmin=0, 
    xmax=df_sorted["annual_visitors_millions"], 
    color="#C77E23", 
    linewidth=2, 
    alpha=0.7
)

# Plot marker dots at endpoints
plt.plot(
    df_sorted["annual_visitors_millions"], 
    df_sorted["destination"], 
    "o", 
    markersize=9, 
    color="#C77E23"
)

plt.xlim(0, 22)
plt.title("Lollipop Chart: Destination Footfall Ranking with Reduced Ink", fontsize=12, weight="bold")
plt.xlabel("Annual Visitors (Millions)", fontsize=11)
plt.ylabel("", fontsize=11)

plt.tight_layout()
plt.show()

### Interpretation for Lollipop Chart
- **Observation:** The lollipop chart highlights the rank order and precise values with a clean, modern aesthetic.
- **Derived Insight:** By reducing visual ink, the reader's attention focuses sharply on the position of each dot along the scale.

## 5. Best Practices & Common Mistakes (Lecture Principles)
**Visualization Best Practices (Slide 19 & 46):**
- **Sort descending:** Never leave default alphabetical order. The reader should not have to rank items by eye.
- **Use horizontal bars** so category names read naturally from left to right.
- **State explicitly how many items were cut** if showing a Top-N view.
- **Add a direct value label** when exact numbers matter to decisions.

**Common Mistakes & Misleading Designs:**
- **Hidden Top-N:** Showing five items out of forty without an 'Other' bar deceives the audience about the remaining volume.
- **Treating a trivial gap as meaningful:** A 1% difference between ranks 1 and 2 is rarely actionable.
- **Ranking on the wrong metric (Slide 18):** Footfall is not revenue or margin. A high-footfall location can generate low average spend per visitor.

## 6. Key Takeaways
1. **Sorting is free analysis:** Always sort ranking charts by value to answer 'what is highest and lowest' instantly.
2. **Horizontal bars are ideal for named entities:** They allow full labels without awkward diagonal angles.
3. **Always include an 'Other' bar when pruning to Top-N** to preserve the true scale of total activity.
4. **Volume is not value:** Never confuse volume (footfall, units) with profitability or commercial success.